# Milestone 8 — Round-trip Test: Qwen3-0.6B

검증 흐름:
1. FP 모델 로드 + baseline generate
2. W4A16 compress → generate
3. save_pretrained → 파일 확인
4. load_pretrained → generate (W4A16과 동일 출력 확인)
5. W8A8 compress + calibration → generate + save

In [ ]:
import sys
sys.path.insert(0, '..')

import os
import json
import tempfile
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

from mini_compressor import Compressor, load_pretrained, save_pretrained
from mini_compressor.schemes import W8A8, W4A16

device = 'cuda' if torch.cuda.is_available() else 'cpu'
MODEL_ID = 'Qwen/Qwen3-0.6B'
print(f'device: {device}')

## 1. FP 모델 로드 + baseline generate

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model_fp = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16).to(device)
model_fp.eval()
print(f'FP model loaded. dtype: {next(model_fp.parameters()).dtype}')

In [ ]:
prompt = 'The key advantage of quantization is'
inputs = tokenizer(prompt, return_tensors='pt').to(device)

with torch.no_grad():
    out_fp = model_fp.generate(**inputs, max_new_tokens=30, do_sample=False)

text_fp = tokenizer.decode(out_fp[0], skip_special_tokens=True)
print(f'[FP] {text_fp}')

## 2. W4A16 compress → generate

In [ ]:
# 새로 로드 (FP 모델 보존용)
model_w4a16 = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16).to(device)
model_w4a16.eval()

compressor_w4a16 = Compressor.from_scheme('w4a16', ignore=['lm_head'])
compressor_w4a16.compress(model_w4a16)  # W4A16은 calibration dataloader 불필요

# FakeQuantLinear로 교체됐는지 확인
from mini_compressor.fake_quant_linear import FakeQuantLinear
q_layers = [(n, m) for n, m in model_w4a16.named_modules() if isinstance(m, FakeQuantLinear)]
print(f'FakeQuantLinear 교체 수: {len(q_layers)}')
print(f'첫 번째 layer: {q_layers[0][0]}, weight_scale shape: {q_layers[0][1].weight_scale.shape}')

In [ ]:
with torch.no_grad():
    out_w4a16 = model_w4a16.generate(**inputs, max_new_tokens=30, do_sample=False)

text_w4a16 = tokenizer.decode(out_w4a16[0], skip_special_tokens=True)
print(f'[W4A16] {text_w4a16}')
print(f'\nFP와 동일: {text_fp == text_w4a16}')

## 3. save_pretrained → 파일 구조 확인

In [ ]:
SAVE_DIR_W4A16 = '/tmp/mini_compressor_w4a16'

compressor_w4a16.save(model_w4a16, SAVE_DIR_W4A16, tokenizer=tokenizer)

print('저장된 파일:', os.listdir(SAVE_DIR_W4A16))

In [ ]:
# quantization_config.json 내용 확인
with open(os.path.join(SAVE_DIR_W4A16, 'quantization_config.json')) as f:
    qconfig = json.load(f)

print(json.dumps(qconfig, indent=2))

## 4. load_pretrained → generate (W4A16 저장본과 비교)

In [ ]:
model_loaded = load_pretrained(SAVE_DIR_W4A16).to(device)
model_loaded.eval()

# FakeQuantLinear가 복원됐는지 확인
loaded_q_layers = [(n, m) for n, m in model_loaded.named_modules() if isinstance(m, FakeQuantLinear)]
print(f'로드 후 FakeQuantLinear 수: {len(loaded_q_layers)}')

# scale 값이 복원됐는지 확인
orig_scale = q_layers[0][1].weight_scale
loaded_scale = loaded_q_layers[0][1].weight_scale
print(f'weight_scale 복원 일치: {torch.allclose(orig_scale.cpu(), loaded_scale.cpu())}')

In [ ]:
inputs_loaded = tokenizer(prompt, return_tensors='pt').to(device)

with torch.no_grad():
    out_loaded = model_loaded.generate(**inputs_loaded, max_new_tokens=30, do_sample=False)

text_loaded = tokenizer.decode(out_loaded[0], skip_special_tokens=True)
print(f'[W4A16 저장 후 로드] {text_loaded}')
print(f'\nW4A16 원본과 동일: {text_w4a16 == text_loaded}')

## 5. W8A8 compress + calibration → save + load

In [ ]:
model_w8a8 = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16).to(device)
model_w8a8.eval()

# 간단한 calibration 데이터
calib_texts = [
    'The quick brown fox jumps over the lazy dog.',
    'Quantization reduces model size by representing weights in lower precision.',
    'Large language models require significant computational resources.',
    'Neural networks learn representations through gradient descent.',
    'Attention mechanisms allow models to focus on relevant input tokens.',
]

calib_inputs = [
    {k: v.to(device) for k, v in tokenizer(t, return_tensors='pt').items()}
    for t in calib_texts
]

compressor_w8a8 = Compressor.from_scheme('w8a8', ignore=['lm_head'])
compressor_w8a8.compress(model_w8a8, dataloader=calib_inputs)

w8a8_layers = [(n, m) for n, m in model_w8a8.named_modules() if isinstance(m, FakeQuantLinear)]
print(f'W8A8 FakeQuantLinear 수: {len(w8a8_layers)}')
print(f'input_scale 복원: {w8a8_layers[0][1].input_scale}')

In [ ]:
with torch.no_grad():
    out_w8a8 = model_w8a8.generate(**inputs, max_new_tokens=30, do_sample=False)

text_w8a8 = tokenizer.decode(out_w8a8[0], skip_special_tokens=True)
print(f'[W8A8] {text_w8a8}')

In [ ]:
SAVE_DIR_W8A8 = '/tmp/mini_compressor_w8a8'
compressor_w8a8.save(model_w8a8, SAVE_DIR_W8A8, tokenizer=tokenizer)

with open(os.path.join(SAVE_DIR_W8A8, 'quantization_config.json')) as f:
    print(json.dumps(json.load(f), indent=2))

In [ ]:
model_w8a8_loaded = load_pretrained(SAVE_DIR_W8A8).to(device)
model_w8a8_loaded.eval()

with torch.no_grad():
    out_w8a8_loaded = model_w8a8_loaded.generate(**inputs, max_new_tokens=30, do_sample=False)

text_w8a8_loaded = tokenizer.decode(out_w8a8_loaded[0], skip_special_tokens=True)
print(f'[W8A8 저장 후 로드] {text_w8a8_loaded}')
print(f'\nW8A8 원본과 동일: {text_w8a8 == text_w8a8_loaded}')

## 결과 요약

In [ ]:
print('=' * 60)
print('[FP]                ', text_fp)
print('[W4A16]             ', text_w4a16)
print('[W4A16 load]        ', text_loaded)
print('[W8A8]              ', text_w8a8)
print('[W8A8 load]         ', text_w8a8_loaded)
print('=' * 60)
print(f'W4A16 round-trip 일치: {text_w4a16 == text_loaded}')
print(f'W8A8  round-trip 일치: {text_w8a8 == text_w8a8_loaded}')